# Open Unified Model .pp files using iris

* Demonstrates how to open Unified Model .pp files (native output) using `iris`
* These are located on the JASMIN kscale group workspace - you will need to have access to that
* You can apply here: https://accounts.jasmin.ac.uk/services/group_workspaces/kscale/
* This will **only** work if you have installed `mo_pack`, which is needed to undertand the .pp file format.
* You can do this by opening a terminal within the notebook service:
    * + button near the top left
    * Other->Terminal
    * `conda activate hk26_env` (or whatever env you are using)
    * `conda install mo_pack`
    * You then need to restart your kernel

In [ ]:
from collections import defaultdict
from pathlib import Path

import cartopy.crs as ccrs
import iris
import iris.plot as iplt
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

In [ ]:
# This is the main location for the simulation output.
dy3dir = Path('/gws/nopw/j04/kscale/DYAMOND3_reruns')
# We ran out of space, so some output is located under scratch.
dy3dir_scratch = Path('/work/scratch-pw5/rwjones/kscale/DYAMOND3_reruns')

In [ ]:
def find_sims(dirs):
    """Returns a dict of sims, where values are another dict containing basedir, is_global and on_scratch"""
    sims = {}
    for d in dirs:
        for first_ppa in sorted(d.glob('*/*/field.pp/apvera.pp/*.apvera_20200120T0000Z.pp')):
            simkey = '.'.join(first_ppa.name.split('.')[:-2])
            basedir = first_ppa.parent.parent.parent
            is_global = 'glm' in simkey
            on_scratch = 'gws' not in str(first_ppa)
            sims[simkey] = {'basedir': basedir, 'is_global': is_global, 'on_scratch': on_scratch}
    return sims


sims = find_sims([dy3dir, dy3dir_scratch])

In [ ]:
for simkey in sorted(sims):
    siminfo = sims[simkey]
    print(f'{simkey}:')
    for k, v in siminfo.items():
        print(f'  {k:<11} = {v}')

In [ ]:
def _parse_date_from_pp_path(path):
    datestr = path.stem.split('.')[-1].split('_')[1]
    if datestr[-1] == 'Z':
        return pd.to_datetime(datestr, format="%Y%m%dT%H%MZ")
    else:
        return pd.to_datetime(datestr, format="%Y%m%dT%H")


def find_dyamond3_pp_dates_to_paths(basedir):
    """Search for pp_paths with a specific date (N.B. filename sensitive)."""
    pp_paths = sorted(basedir.glob('field.pp/apve*/**/*.pp'))
    pp_paths = [p for p in pp_paths if p.is_file()]
    dates_to_paths = defaultdict(dict)
    for path in pp_paths:
        if 'apvere' in path.stem:
            continue
        stream = path.name.split('.')[-2][5]
        dates_to_paths[_parse_date_from_pp_path(path)][stream] = path
    # Only keep completed downloads.
    dates_to_paths = {k: v for k, v in dates_to_paths.items() if len(v) == 4}
    return dates_to_paths

In [ ]:
siminfo = sims['glm.n2560_RAL3p3_tuned']

dates_to_paths = find_dyamond3_pp_dates_to_paths(siminfo['basedir'])
# dates_to_paths is a mapping, keyed by a pd.Timestamp, that gives all .pp files for a given date.
dates_to_paths[pd.Timestamp('2020-01-20 00:00')]

In [ ]:
# We can now construct an iris cube list, loading all streams or just a single stream.
# You can apply a constraint to just load a single cube here
# Load all cubes for time:
# cubes = iris.load(dates_to_paths[pd.Timestamp('2020-01-20 00:00')].values())
# Load single stream:
cubes = iris.load(dates_to_paths[pd.Timestamp('2020-01-20 00:00')]['a'])

In [ ]:
cubes

In [ ]:
tas = cubes.extract_cube('air_temperature')

In [ ]:
tas

In [ ]:
fig = plt.figure(figsize=(10, 5))
projection = ccrs.PlateCarree(central_longitude=0)
crs = ccrs.PlateCarree()
axes = fig.add_subplot(projection=projection)
iplt.pcolormesh(tas[1], cmap='RdYlBu_r')
axes.coastlines(resolution="10m", color='k')
plt.colorbar(orientation='horizontal', shrink=0.7, pad=0.042)

In [ ]:
# We can convert into xarray if we prefer:
tas_da = xr.DataArray.from_iris(tas)

In [ ]:
tas_da